# TFR — 03: ROI × Band × Window Statistics

Extracts mean power per ROI × frequency band × time window and runs
paired tests with FDR correction.

For spatiotemporal cluster tests across the full scalp × time × frequency space,
see notebook 04.

In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, extract_mean_band_power, run_tfr_paired_tests

# ── Update these paths ──
cfg     = load_config('../../configs/your_experiment.yaml')
cfg_tfr = load_config('../../configs/your_tfr_analysis.yaml')

print("Setup OK")

In [ ]:
# ── Define ROIs, frequency bands, and time windows ──
# ── Update based on your montage and hypotheses ──
rois = {
    'frontal':    ['Fz', 'F3', 'F4'],
    'central':    ['Cz', 'C3', 'C4'],
    'posterior':  ['Pz', 'P3', 'P4', 'Oz'],
}

freq_bands = {
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
}

time_windows = {
    'early': (0.1, 0.3),
    'mid':   (0.3, 0.6),
    'late':  (0.6, 1.0),
}

In [ ]:
# ── Extract mean band power ──
df = extract_mean_band_power(
    cfg, cfg_tfr,
    window_name='your_window',
    rois=rois,
    freq_bands=freq_bands,
    time_windows=time_windows,
    conditions=['condition_a', 'condition_b'],
)
print(f"Extracted {len(df)} rows")
df.head()

In [ ]:
# ── Save CSV for external software ──
from eeg_toolkit.io import get_analysis_dir

out_dir = get_analysis_dir(cfg) / 'group_results' / 'tfr' / 'stats'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'mean_band_power.csv'
df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

In [ ]:
# ── Paired tests: condition_a vs condition_b ──
stats_df = run_tfr_paired_tests(df, contrast=('condition_a', 'condition_b'))
stats_df

In [ ]:
# ── Optional: subset to a specific band/ROI ──
df_subset = df[
    (df['freq_band'] == 'alpha') &
    (df['roi'].isin(['posterior']))
]
run_tfr_paired_tests(df_subset, contrast=('condition_a', 'condition_b'))